In [1]:
import requests
import pandas as pd
import time

print("Aguarde, isso pode levar alguns minutos...")

df_avioes = pd.read_csv('thor_wwii_aircraft_gloss.csv')
lista_avioes = df_avioes['full_name'].dropna().unique()

dict_resultados = {}
url_api = "https://en.wikipedia.org/w/api.php"

# Identificação pro wikipedia, email falso mas ele acredita :D
headers = {
    'User-Agent': 'ProjetoDashboardPUC/2.1 (projeto_academico@puc.br)'
}

#Vai procurar o avião pelo nome
for aviao in lista_avioes:
    termo_busca = f"{aviao} aircraft"
    print(f"Buscando: {aviao}...")
    
    sucesso = False
    tentativas = 0
    max_tentativas = 3 # Se falhar 3 vezes ele desiste
    
    while not sucesso and tentativas < max_tentativas:
        parametros = {
            "action": "query",
            "format": "json",
            "generator": "search",       
            "gsrsearch": termo_busca,    
            "gsrlimit": 1,               
            "prop": "extracts",          
            "exintro": True, 
            "explaintext": True
        }
        
        try:
            # Gap de 10 seg com a wikipedia pra não travar o codigo
            resposta = requests.get(url_api, params=parametros, headers=headers, timeout=10)
            
            if resposta.status_code == 200:
                dados_json = resposta.json()
                paginas = dados_json.get('query', {}).get('pages', {})
                
                #Extrai o conteúdo achado
                if paginas:
                    id_pagina = list(paginas.keys())[0]
                    texto_completo = paginas[id_pagina].get('extract', '')

                    #Corta textões ao meio e salva as 2 primeiras frases
                    if len(texto_completo) > 10:
                        resumo = '. '.join(texto_completo.split('. ')[:2]) + '.'
                        dict_resultados[aviao] = resumo
                    else:
                        dict_resultados[aviao] = "Texto insuficiente na base."
                    sucesso = True # Deu certo
                    
                else:
                    # O não achou nada. Tenta de novo sem a palavra "aircraft"
                    if tentativas == 0:
                        print(f"  -> Reajustando busca para: {aviao}")
                        termo_busca = aviao
                        tentativas += 1
                        time.sleep(1)
                    else:
                        dict_resultados[aviao] = "Nenhum registro encontrado."
                        sucesso = True
            #Impede a wiki de bloquear meu ip           
            elif resposta.status_code == 429: 
                print("  -> Servidor sobrecarregado (Erro 429). Espere por 5 segundos...")
                time.sleep(5)
                tentativas += 1
                
            else:
                print(f"  -> Falha no servidor (HTTP {resposta.status_code}). Tentando novamente...")
                time.sleep(2)
                tentativas += 1
                if tentativas == max_tentativas:
                    dict_resultados[aviao] = f"Falha persistente na API (Erro {resposta.status_code})."
        #Não mata o codigo se a net cair durante o processo            
        except requests.exceptions.RequestException as e:
            print(f"  -> Erro de conexão de rede. Tentando de novo...")
            time.sleep(2)
            tentativas += 1
            if tentativas == max_tentativas:
                dict_resultados[aviao] = "Erro de conexão persistente."
                
    # Pausa de segurança padrão entre os aviões
    time.sleep(1)

df_avioes['lore_historico'] = df_avioes['full_name'].map(dict_resultados)
df_avioes.to_csv('thor_aircraft_intel.csv', index=False)

print("\nVarredura concluída! Arquivo 'thor_aircraft_intel.csv' atualizado.")

Aguarde, isso pode levar alguns minutos...
Buscando: Douglas A-20 Havoc...
Buscando: Douglass A-24 Banshee...
Buscando: Douglas A-26 Invader...
Buscando: North American A-36 Apache (Invader)...
Buscando: Fairey Albacore...
Buscando: Hawker Audax...
Buscando: B-17 Flying Fortress...
Buscando: B-24 Liberator...
Buscando: B-25 Mitchell...
Buscando: Martin B-26 Marauder...
Buscando: Boeing B-29 Superfortress...
Buscando: B-32 Dominator...
Buscando: Martin Baltimore...
Buscando: Fairey Battle...
Buscando: Bristol Beaufighter...
Buscando: Bristol Beaufort...
Buscando: Bristol Blenheim...
Buscando: Bristol Bombay...
Buscando: PBY Catalina...
Buscando: Vought F4U Corsair...
Buscando: Gloster Gladiator...
Buscando: Handley Page Halifax...
Buscando: Handley Page HP.52 Hampden...
Buscando: Hawker Hardy...
Buscando: Lockheed Hudson...
Buscando: Hawker Hurricane...
Buscando: Avro Lancaster...
Buscando: Junkers Ju 86...
Buscando: Westland Lysander...
Buscando: Avro Manchester...
Buscando: Martin Mar